In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPU: []


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATA_DIR = Path(
    "/content/drive/MyDrive/Skin-Lesion-Segmentation/data"
)

print("Files found:")

for file in DATA_DIR.glob("*.npy"):
    print(file.name)

Files found:
X_test.npy
X_train.npy
X_val.npy
Y_test.npy
Y_train.npy
Y_val.npy


In [ ]:
import numpy as np

X_train = np.load(DATA_DIR / "X_train.npy")
Y_train = np.load(DATA_DIR / "Y_train.npy")

X_val = np.load(DATA_DIR / "X_val.npy")
Y_val = np.load(DATA_DIR / "Y_val.npy")

X_test = np.load(DATA_DIR / "X_test.npy")
Y_test = np.load(DATA_DIR / "Y_test.npy")

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)

X_train: (720, 256, 256, 3)
Y_train: (720, 256, 256, 1)
X_val: (90, 256, 256, 3)
Y_val: (90, 256, 256, 1)
X_test: (90, 256, 256, 3)
Y_test: (90, 256, 256, 1)


In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.models import Model

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        activation="relu"
    )(x)

    return x

In [ ]:
inputs = layers.Input(shape=(256, 256, 3))

# Encoder
c1 = conv_block(inputs, 64)
p1 = layers.MaxPooling2D(pool_size=(2, 2))(c1)

c2 = conv_block(p1, 128)
p2 = layers.MaxPooling2D(pool_size=(2, 2))(c2)

c3 = conv_block(p2, 256)
p3 = layers.MaxPooling2D(pool_size=(2, 2))(c3)

c4 = conv_block(p3, 512)
p4 = layers.MaxPooling2D(pool_size=(2, 2))(c4)

# Bottleneck
c5 = conv_block(p4, 1024)

# Decoder
u6 = layers.Conv2DTranspose(
    512,
    kernel_size=2,
    strides=2,
    padding="same"
)(c5)

u6 = layers.concatenate([u6, c4])
c6 = conv_block(u6, 512)

u7 = layers.Conv2DTranspose(
    256,
    kernel_size=2,
    strides=2,
    padding="same"
)(c6)

u7 = layers.concatenate([u7, c3])
c7 = conv_block(u7, 256)

u8 = layers.Conv2DTranspose(
    128,
    kernel_size=2,
    strides=2,
    padding="same"
)(c7)

u8 = layers.concatenate([u8, c2])
c8 = conv_block(u8, 128)

u9 = layers.Conv2DTranspose(
    64,
    kernel_size=2,
    strides=2,
    padding="same"
)(c8)

u9 = layers.concatenate([u9, c1])
c9 = conv_block(u9, 64)

# Output
outputs = layers.Conv2D(
    1,
    kernel_size=1,
    activation="sigmoid"
)(c9)

model = Model(
    inputs=inputs,
    outputs=outputs
)

print("U-Net created!")

U-Net created!


In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 32,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 32, 32,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 32, 32,    │  2,359,808 │ conv2d_6[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 16, 16,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 16, 16,    │  4,719,616 │ max_pooling2d_3[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 16, 16,    │  9,438,208 │ conv2d_8[0][0]    │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 32, 32,    │  2,097,664 │ conv2d_9[0][0]    │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32, 32,    │          0 │ conv2d_transpose

 Total params: 31,031,745 (118.38 MB)

 Trainable params: 31,031,745 (118.38 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true_f * y_pred_f)

    dice = (
        2.0 * intersection + smooth
    ) / (
        tf.reduce_sum(y_true_f) +
        tf.reduce_sum(y_pred_f) +
        smooth
    )

    return dice


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred
    )

    bce = tf.reduce_mean(bce)

    return bce + dice_loss(y_true, y_pred)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=combined_loss,
    metrics=[
        dice_coefficient,
        "binary_accuracy"
    ]
)

print("Model compiled successfully!")

Model compiled successfully!


In [ ]:
import tensorflow as tf
import os

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_coefficient",
        mode="max",
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/Skin-Lesion-Segmentation/models/unet_skin_lesion_best.keras",
        monitor="val_dice_coefficient",
        mode="max",
        save_best_only=True
    )
]

In [ ]:
history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=20,
    batch_size=8,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 5186s 58s/step - binary_accuracy: 0.7603 - dice_coefficient: 0.4111 - loss: 1.1172 - val_binary_accuracy: 0.8266 - val_dice_coefficient: 0.4352 - val_loss: 0.9948 - learning_rate: 1.0000e-04
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 5191s 58s/step - binary_accuracy: 0.8428 - dice_coefficient: 0.5862 - loss: 0.8363 - val_binary_accuracy: 0.8601 - val_dice_coefficient: 0.6379 - val_loss: 0.7554 - learning_rate: 1.0000e-04
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 5212s 58s/step - binary_accuracy: 0.8840 - dice_coefficient: 0.7010 - loss: 0.6404 - val_binary_accuracy: 0.9074 - val_dice_coefficient: 0.7442 - val_loss: 0.5449 - learning_rate: 1.0000e-04
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 5217s 58s/step - binary_accuracy: 0.8950 - dice_coefficient: 0.7423 - loss: 0.5590 - val_binary_accuracy: 0.9092 - val_dice_coefficient: 0.7641 - val_loss: 0.4892 - learning_rate: 1.0000e-04
Epoch 5/20
13/90 ━━━━━━━━━━━━━━━━━━━━ 1:11:00 55s/step - binary_accuracy: 0.8920 - d

In [ ]:
MODEL_DIR = "/content/drive/MyDrive/Skin-Lesion-Segmentation/models"
ModelCheckpoint(
    os.path.join(
        MODEL_DIR,
        "unet_skin_lesion_best.keras"
    ),
    monitor="val_dice_coefficient",
    mode="max",
    save_best_only=True
)

In [2]:
#we are starting our training from the model which is saved till 4th epoch
import os

MODEL_PATH = "/content/drive/MyDrive/Skin-Lesion-Segmentation/models/unet_skin_lesion_best.keras"

print(os.path.exists(MODEL_PATH))

True


## Training Summary

The U-Net was trained for up to 20 epochs using the ISIC 2016 lesion segmentation dataset.

The model used:
- Input resolution: 256 × 256 × 3
- Architecture: U-Net
- Parameters: ~31 million
- Optimizer: Adam
- Initial learning rate: 1e-4
- Loss: Binary Cross-Entropy + Dice Loss
- Batch size: 8
- Early stopping based on validation Dice
- Model checkpointing based on the best validation Dice

Training was computationally intensive because of the large U-Net architecture. The training runtime was interrupted after four complete epochs and part of the fifth epoch due to Colab GPU usage limitations.

The best completed checkpoint was obtained at Epoch 4.

| Epoch | Training Dice | Validation Dice | Validation Loss |
|------:|--------------:|----------------:|----------------:|
| 1 | 0.4111 | 0.4352 | 0.9948 |
| 2 | 0.5862 | 0.6379 | 0.7554 |
| 3 | 0.7010 | 0.7442 | 0.5449 |
| 4 | 0.7423 | 0.7641 | 0.4892 |

The best model achieved a validation Dice coefficient of **0.7641** and was successfully saved using ModelCheckpoint. This checkpoint is used for the final evaluation on the held-out test set.